# PAADU Phase 6 — Reconstructed Hybrid GraphRAG + NLI + CHARM Boundary Audit

**One-click Colab workflow:** Runtime → Run all. Produces `PAADU_Phase6_Results.zip` and downloads it automatically.

**Research boundaries:** Hybrid retrieval here is a **reconstruction** inspired by the student report, not execution of the full original student repository. CHARM is the original network-anomaly implementation and **is not a text-hallucination detector**. This phase audits both domains separately and integrates retrieval→generation→verification→conservative correction; it does **not** claim a validated CHARM adaptation. HotpotQA's 30 previously selected questions are reused for a **diagnostic comparison**, not a new independent test. Gold answers are used **only for evaluation**, never retrieval ranking, generation prompts, or correction decisions.

**Expected runtime:** approximately 15–45 minutes depending on CPU/GPU/network. Download requires internet. If a required stage fails, `STATUS.json` records incomplete results instead of fabricating scores.

In [ ]:
!pip -q install "transformers>=4.44,<5" "sentencepiece>=0.2" "scikit-learn>=1.3" "pandas>=2.0" "requests>=2.31" "networkx>=3.0" "torch>=2.1"

In [ ]:
import os,sys,re,json,io,zipfile,hashlib,time,traceback,subprocess,random
from pathlib import Path
from datetime import datetime,timezone
import numpy as np,pandas as pd,requests,networkx as nx,torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer,AutoModelForSeq2SeqLM,AutoModelForSequenceClassification
from IPython.display import display
ROOT=Path('/content/PAADU_Phase6') if Path('/content').exists() else Path('/mnt/data/PAADU_Phase6')
ROOT.mkdir(parents=True,exist_ok=True)
N_QUESTIONS=30; SEED=20261106; random.seed(SEED); np.random.seed(SEED)
STATUS={'phase':'6','complete':False,'started_utc':datetime.now(timezone.utc).isoformat(),'checks':{},'errors':[],'notes':['Student GraphRAG reconstructed; not the original complete repo','CHARM original network-only; no text adaptation','Repeated 30-question diagnostic set; not independent validation']}
def checkpoint(name,fn):
 try:
  result=fn();STATUS['checks'][name]='PASS';print('PASS:',name);return result
 except Exception as e:
  STATUS['checks'][name]='FAIL';STATUS['errors'].append({'stage':name,'error':str(e),'traceback':traceback.format_exc()[-5000:]});print('FAIL:',name,repr(e));return None


In [ ]:
# Embedded prior research archive: CHARM_XAI_Phase1_Research.zip; SHA-256 checked before use.
import base64
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='ce6f37dd4b34f77b0b63fb69407feb4402a40f35831d3cf5a47f5861fff0ee54'
(ROOT/'CHARM_XAI_Phase1_Research.zip').write_bytes(payload)
del payload
print('Restored CHARM_XAI_Phase1_Research.zip')

In [ ]:
# Embedded prior research archive: PAADU_Phase5_1_Results.zip; SHA-256 checked before use.
import base64
payload=base64.b64decode('...')
assert hashlib.sha256(payload).hexdigest()=='94c1554f1dd782b4b03cbea63a092cfa53eedab94584a5cc7100768d1e553bbb'
(ROOT/'PAADU_Phase5_1_Results.zip').write_bytes(payload)
del payload
print('Restored PAADU_Phase5_1_Results.zip')

## 1. Load the exact HotpotQA distractor development file and reproduce Phase 5 selection

In [ ]:
DATA_URLS=[
'https://huggingface.co/datasets/RAGLAB/data/resolve/main/eval_datasets/HotPotQA/hotpot_dev_distractor_v1.json?download=true',
'https://huggingface.co/datasets/namlh2004/hotpotqa/resolve/main/hotpot_dev_distractor_v1.json?download=true']
EXPECTED_SHA='4e9ecb5c8d3b719f624d66b60f8d56bf227f03914f5f0753d6fa1b359d7104ea'
def load_data():
 failures=[]
 for url in DATA_URLS:
  try:
   response=requests.get(url,timeout=120);response.raise_for_status();raw=response.content
   sha=hashlib.sha256(raw).hexdigest()
   if sha!=EXPECTED_SHA:raise ValueError('SHA-256 mismatch: '+sha)
   items=json.loads(raw)
   assert isinstance(items,list) and len(items)>N_QUESTIONS
   (ROOT/'DATASET_SHA256.txt').write_text(sha+'\n'+url+'\n')
   STATUS['dataset_url']=url;STATUS['dataset_sha256']=sha
   return items
  except Exception as e:failures.append({'url':url,'error':str(e)})
 raise RuntimeError('All checksum-verified dataset URLs failed: '+json.dumps(failures))
DATA=checkpoint('dataset',load_data)
def select_questions(items):
 # Same seed+ID SHA-256 rule as Phase 5. Recheck IDs against archived Phase 5 results.
 return sorted(items,key=lambda x:hashlib.sha256((str(SEED)+str(x['_id'])).encode()).hexdigest())[:N_QUESTIONS]
BENCH=select_questions(DATA) if DATA else []
print('Selected:',len(BENCH))

## 2. Reconstructed hybrid retrieval (TF-IDF + NetworkX paragraph-title graph)

In [ ]:
def paragraphs(item):
 return [{'title':str(t),'text':' '.join(map(str,sents)),'sentences':[str(s) for s in sents]} for t,sents in item['context']]
def build_hybrid(item):
 docs=paragraphs(item)
 corpus=[d['title']+' '+d['text'] for d in docs]
 vec=TfidfVectorizer(stop_words='english',ngram_range=(1,2),max_features=12000)
 mat=vec.fit_transform(corpus)
 titles=[d['title'].lower() for d in docs]
 graph=nx.Graph();graph.add_nodes_from(range(len(docs)))
 # Graph edges are grounded in document title mentions, not answer labels.
 for i,d in enumerate(docs):
  text=d['text'].lower()
  for j,t in enumerate(titles):
   if i!=j and len(t)>=4 and t in text:graph.add_edge(i,j)
 return docs,vec,mat,graph
def retrieve(item,mode='hybrid',k=4):
 docs,vec,mat,graph=build_hybrid(item)
 q=vec.transform([item['question']]);scores=cosine_similarity(q,mat).ravel()
 if mode=='hybrid':
  # Only boost graph neighbors of query-relevant anchors; graph links alone never suffice.
  anchors=np.argsort(-scores)[:2];boost=np.zeros(len(docs))
  for anchor in anchors:
   for j in graph.neighbors(int(anchor)):
    boost[j]=max(boost[j],float(scores[anchor])*0.22)
  scores=scores+boost
 selected=np.argsort(-scores)[:min(k,len(docs))]
 return [{'title':docs[int(i)]['title'],'text':docs[int(i)]['text'],'sentences':docs[int(i)]['sentences'],'score':float(scores[i])} for i in selected],{'nodes':len(graph),'edges':graph.number_of_edges()}
def test_retrieval():
 item={'question':'What does UDP provide?','context':[['UDP',['UDP is connectionless.']],['TCP',['TCP is connection-oriented.']]]}
 a,_=retrieve(item,'hybrid',2);assert any(x['title']=='UDP' for x in a)
 return True
checkpoint('retrieval_unit',test_retrieval)

## 3. Load real generation and NLI models (no fake fallback)

In [ ]:
GEN_ID='google/flan-t5-small';NLI_ID='cross-encoder/nli-MiniLM2-L6-H768'
DEVICE='cuda' if torch.cuda.is_available() else 'cpu';print('Device:',DEVICE)
def load_models():
 global gen_tok,gen_model,nli_tok,nli_model,ID2LABEL
 gen_tok=AutoTokenizer.from_pretrained(GEN_ID)
 gen_model=AutoModelForSeq2SeqLM.from_pretrained(GEN_ID).to(DEVICE).eval()
 nli_tok=AutoTokenizer.from_pretrained(NLI_ID)
 nli_model=AutoModelForSequenceClassification.from_pretrained(NLI_ID).to(DEVICE).eval()
 ID2LABEL={int(k):str(v).lower() for k,v in nli_model.config.id2label.items()}
 assert {'contradiction','entailment','neutral'}==set(ID2LABEL.values()),ID2LABEL
 STATUS['models']={'generator':GEN_ID,'nli':NLI_ID,'nli_id2label':ID2LABEL,'device':DEVICE}
 return True
MODELS=checkpoint('models',load_models)

## 4. Evidence-aware verification and conservative policy

In [ ]:
def generate_answer(question,evidence):
 context='\n'.join(x['title']+': '+x['text'] for x in evidence)
 prompt='Answer the question using only the context. Give a short answer.\nContext: '+context[:3200]+'\nQuestion: '+question+'\nAnswer:'
 tokens=gen_tok(prompt,return_tensors='pt',truncation=True,max_length=512).to(DEVICE)
 with torch.no_grad():ids=gen_model.generate(**tokens,max_new_tokens=36,num_beams=3,do_sample=False)
 return gen_tok.decode(ids[0],skip_special_tokens=True).strip()
def topical_score(claim,sentence,question):
 stop={'the','and','for','with','that','this','what','which','where','when','does','both','have','from','into','was','were','are','who','how','did','its','their','they','than','then'}
 toks=lambda s:set(re.findall(r'[a-z0-9]+',s.lower()))-stop
 c=toks(claim+' '+question);s=toks(sentence)
 return len(c&s)/max(1,min(len(c),len(s)))
def verify_answer(question,answer,evidence):
 if not answer:return {'label':'INSUFFICIENT_EVIDENCE','evidence':None,'score':0.0,'topical_score':0.0}
 sentences=[{'title':d['title'],'sentence':s} for d in evidence for s in d['sentences'] if len(s.strip())>5]
 ranked=sorted([(topical_score(answer,x['sentence'],question),x) for x in sentences],key=lambda z:-z[0])
 candidates=[(rel,x) for rel,x in ranked[:8] if rel>=0.15]
 if not candidates:return {'label':'INSUFFICIENT_EVIDENCE','evidence':None,'score':0.0,'topical_score':0.0}
 pairs=nli_tok([x['sentence'] for _,x in candidates],[answer]*len(candidates),padding=True,truncation=True,max_length=256,return_tensors='pt').to(DEVICE)
 with torch.no_grad():probs=torch.softmax(nli_model(**pairs).logits,dim=-1).cpu().numpy()
 hits=[]
 for (rel,x),p in zip(candidates,probs):
  label=ID2LABEL[int(p.argmax())];hits.append({'label':label,'prob':float(max(p)),'entail':float(p[next(k for k,v in ID2LABEL.items() if v=='entailment')]),'contra':float(p[next(k for k,v in ID2LABEL.items() if v=='contradiction')]),'rel':rel,'evidence':x})
 ent=sorted(hits,key=lambda h:h['entail']*h['rel'],reverse=True)[0]
 con=sorted(hits,key=lambda h:h['contra']*h['rel'],reverse=True)[0]
 # High contradiction threshold and no competing strong entailment.
 if con['contra']>=0.98 and con['rel']>=0.30 and ent['entail']<0.80:
  best=con;label='CONTRADICTED'
 elif ent['entail']>=0.85 and ent['rel']>=0.20:
  best=ent;label='SUPPORTED'
 else:best=ent if ent['entail']>=con['contra'] else con;label='INSUFFICIENT_EVIDENCE'
 return {'label':label,'evidence':best['evidence'],'score':float(best['contra'] if label=='CONTRADICTED' else best['entail']),'topical_score':float(best['rel'])}
def conservative_correction(answer,verification):
 # No hallucinated 'correction': only remove a claim when a strong contradiction is demonstrated.
 # Abstention is a safety action, not evidence of improved answer accuracy.
 if verification['label']=='CONTRADICTED':return 'Insufficient verified evidence'
 return answer

def normalized(s):return re.sub(r'\s+',' ',re.sub(r'[^a-z0-9 ]',' ',str(s).lower())).strip()
def em(pred,gold):return int(normalized(pred)==normalized(gold))
def f1(pred,gold):
 from collections import Counter
 a=normalized(pred).split();b=normalized(gold).split()
 if not a or not b:return float(a==b)
 common=sum((Counter(a)&Counter(b)).values())
 return 2*common/(len(a)+len(b))
assert conservative_correction('yes',{'label':'INSUFFICIENT_EVIDENCE'})=='yes'
assert conservative_correction('yes',{'label':'CONTRADICTED'})!='yes'
STATUS['checks']['correction_unit']='PASS' 

## 5. Original CHARM domain audit — no text-hallucination claims

In [ ]:
def audit_charm():
 archive=ROOT/'CHARM_XAI_Phase1_Research.zip';dest=ROOT/'charm_original'
 dest.mkdir(exist_ok=True)
 with zipfile.ZipFile(archive) as z:z.extractall(dest)
 # Execute original project's smoke test unchanged in a separate subprocess.
 # It trains on synthetic network blocks; this is a wiring test, not CICIoT2023 validation.
 env=dict(os.environ);env['PYTHONPATH']=str(dest/'project')
 cmd=[sys.executable,str(dest/'project'/'smoke_test.py')]
 run=subprocess.run(cmd,cwd=dest/'project',env=env,text=True,capture_output=True,timeout=240)
 (ROOT/'CHARM_SMOKE_STDOUT.txt').write_text(run.stdout+'\nSTDERR:\n'+run.stderr)
 if run.returncode!=0:raise RuntimeError('CHARM smoke test failed: '+run.stderr[-1600:])
 assert 'PASS:' in run.stdout
 return {'test':'original network CHARM smoke test','exit_code':run.returncode,'stdout':run.stdout,'text_hallucination_adaptation':'NOT_IMPLEMENTED'}
CHARM_AUDIT=checkpoint('original_charm_network_smoke',audit_charm)
if CHARM_AUDIT:(ROOT/'CHARM_AUDIT.json').write_text(json.dumps(CHARM_AUDIT,indent=2))

## 6. Run paired retrieval baselines and verification on the same 30 questions

In [ ]:
def prior_ids():
 with zipfile.ZipFile(ROOT/'PAADU_Phase5_1_Results.zip') as outer:
  with zipfile.ZipFile(io.BytesIO(outer.read('PAADU_Phase5_Fixed_Results_Original.zip'))) as inner:
   return set(pd.read_csv(io.BytesIO(inner.read('ANSWER_EVALUATION.csv')))['id'].astype(str))
PRIOR_IDS=prior_ids()
if BENCH:
 assert len(BENCH)==N_QUESTIONS
 SELECTED_IDS={str(x['_id']) for x in BENCH}
 STATUS['same_questions_as_phase5']=SELECTED_IDS==PRIOR_IDS
 if SELECTED_IDS!=PRIOR_IDS:
  STATUS['errors'].append({'stage':'selection_audit','error':'Phase 5 selection differs; comparison is not paired'})
 print('Paired IDs identical to Phase 5:',STATUS['same_questions_as_phase5'])
RECORDS=[]
if BENCH and MODELS:
 for i,item in enumerate(BENCH):
  try:
   result={'id':str(item['_id']),'question':item['question'],'gold_answer':str(item['answer'])}
   for mode in ['tfidf','hybrid']:
    evidence,graph=retrieve(item,mode)
    answer=generate_answer(item['question'],evidence)
    ver=verify_answer(item['question'],answer,evidence)
    corrected=conservative_correction(answer,ver)
    result[mode]={'answer':answer,'verification':ver,'corrected':corrected,'em':em(answer,item['answer']),'f1':f1(answer,item['answer']),'corrected_em':em(corrected,item['answer']),'corrected_f1':f1(corrected,item['answer']),'graph':graph,'retrieved_titles':[x['title'] for x in evidence]}
   RECORDS.append(result)
   print(f'{i+1}/{len(BENCH)}',result['id'],'tfidf EM',result['tfidf']['em'],'hybrid EM',result['hybrid']['em'])
  except Exception as e:
   STATUS['errors'].append({'stage':'question','id':str(item.get('_id')),'error':str(e),'traceback':traceback.format_exc()[-2000:]})
STATUS['checks']['paired_benchmark']='PASS' if len(RECORDS)==N_QUESTIONS and STATUS.get('same_questions_as_phase5') else 'FAIL'
print('Completed paired records:',len(RECORDS))

## 7. Metrics, provenance, uncertainty, status, and final ZIP

In [ ]:
def metrics_for(mode):
 rows=[r[mode] for r in RECORDS]
 if not rows:return None
 return {'n':len(rows),'raw_em':float(np.mean([r['em'] for r in rows])),'raw_token_f1':float(np.mean([r['f1'] for r in rows])),'corrected_em':float(np.mean([r['corrected_em'] for r in rows])),'corrected_token_f1':float(np.mean([r['corrected_f1'] for r in rows])),'abstentions':sum(r['corrected']!=r['answer'] for r in rows),'nli_counts':{lab:sum(r['verification']['label']==lab for r in rows) for lab in ['SUPPORTED','CONTRADICTED','INSUFFICIENT_EVIDENCE']}}
METRICS={'tfidf':metrics_for('tfidf'),'hybrid':metrics_for('hybrid'),'n_questions':len(RECORDS),'target_questions':N_QUESTIONS,'paired_to_phase5':STATUS.get('same_questions_as_phase5',False),'independent_benchmark':False,'charm_text_hallucination_adaptation':False}
STATUS['complete']=all(STATUS['checks'].get(k)=='PASS' for k in ['dataset','retrieval_unit','models','correction_unit','original_charm_network_smoke','paired_benchmark'])
STATUS['finished_utc']=datetime.now(timezone.utc).isoformat()
(ROOT/'STATUS.json').write_text(json.dumps(STATUS,indent=2))
(ROOT/'METRICS.json').write_text(json.dumps(METRICS,indent=2))
(ROOT/'EVALUATION_TRACES.json').write_text(json.dumps(RECORDS,indent=2))
flat=[]
for r in RECORDS:
 for mode in ['tfidf','hybrid']:
  x=r[mode];flat.append({'id':r['id'],'mode':mode,'question':r['question'],'gold_answer':r['gold_answer'],'answer':x['answer'],'corrected':x['corrected'],'em':x['em'],'token_f1':x['f1'],'corrected_em':x['corrected_em'],'corrected_token_f1':x['corrected_f1'],'nli_label':x['verification']['label'],'nli_score':x['verification']['score'],'retrieved_titles':' | '.join(x['retrieved_titles'])})
pd.DataFrame(flat).to_csv(ROOT/'ANSWER_EVALUATION.csv',index=False)
report=f"""# Phase 6 Research Report

**Execution status:** {'COMPLETE' if STATUS['complete'] else 'INCOMPLETE'}

**Paired questions:** {len(RECORDS)}/{N_QUESTIONS}; same question IDs as Phase 5: {STATUS.get('same_questions_as_phase5')}

**TF-IDF baseline:** {json.dumps(METRICS['tfidf'],indent=2)}

**Reconstructed hybrid GraphRAG:** {json.dumps(METRICS['hybrid'],indent=2)}

**Original CHARM smoke test:** {STATUS['checks'].get('original_charm_network_smoke','NOT RUN')} (network telemetry only).

**Scope restrictions:** Student full original code not run; hybrid retrieval is reconstructed. CHARM's original model and network scoring are not adapted to text. NLI classifications lack independently annotated hallucination labels. Reused 30-question set is diagnostic, not an independent test. Small sample sizes and correction abstentions limit conclusions. Gold answers used only for scoring.

**Errors:** {json.dumps(STATUS['errors'],indent=2)}
"""
(ROOT/'PHASE6_FINAL_REPORT.md').write_text(report)
zip_path=ROOT.parent/'PAADU_Phase6_Results.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED) as z:
 for f in sorted(ROOT.iterdir()):
  if f.is_file() and f.name not in ['CHARM_XAI_Phase1_Research.zip','PAADU_Phase5_1_Results.zip']:
   z.write(f,f.name)
 z.write(ROOT/'PAADU_Phase5_1_Results.zip','PAADU_Phase5_1_Results.zip')
print('STATUS:', 'COMPLETE' if STATUS['complete'] else 'INCOMPLETE')
print('Results:',zip_path)
print(report[:2200])
try:
 from google.colab import files
 files.download(str(zip_path))
except ImportError:print('Outside Colab; ZIP saved at',zip_path)
